# 09 · Make your pipeline work beyond COMPAS

Today there's no new technique. The whole class is **yours**, working on **your own pipeline**.

Since week 2, your pipeline has only seen one dataset (COMPAS) and one kind of question (yes/no). Some of its code quietly assumes both. Today you find out which parts. **Pick one option:**

| | What changes | What it tests |
|---|---|---|
| **Option A · new data** | a different dataset (Adult income), still a yes/no question | everything that only works for *COMPAS* |
| **Option B · new kind of target** | COMPAS again, but predicting one of **three** classes | everything that only works for *two classes* |

Each option changes **one** thing, so whatever breaks points straight at a hidden assumption.

This notebook doesn't do the work for you. It gives you **what you need to know** (for A, the data has already been checked for you, so you don't need to explore it) and **the main steps**. The code is yours to write, in your fork.

## 1 · Before you start (both options)

- **Keep COMPAS working.** At the end of today, `python main.py` must still run the COMPAS pipeline exactly as before. Your new problem gets **its own config file** (`config_adult.yaml` or `config_multiclass.yaml`), and `main.py` needs a way to choose it, e.g. `python main.py config_adult.yaml`.
- **Test fast, then run properly.** While fixing things, use `tuning.enabled: false` (or a small `n_trials`). Turn tuning back on for the final runs.
- **Give each config its own `output` folder**, so the scoreboard doesn't mix runs from different problems.
- **Commit often**, with a message saying what you fixed.
- Keep a list as you go: **what broke, why it broke, how you fixed it.** That list is your README section (§4).

## 2 · Option A · New data: Adult income

**The question:** does this person earn more than 50,000 USD a year? The data comes from the 1994 US Census (the UCI *Adult* dataset), subsampled and extended for this course.

| File (in `data/`, next to this notebook) | Rows | Target |
|---|---|---|
| `adult_train.csv` | 6,000 people | yes: `income_over_50k` (1 = yes, 0 = no) |
| `adult_new.csv` | 2,000 new people | no: your final model predicts these |

Copy both files into your repo's `data/` folder. Column descriptions are in `data/ADULT_README.md`.

In [1]:
import pandas as pd

train = pd.read_csv("data/adult_train.csv")
new = pd.read_csv("data/adult_new.csv")
print(f"train: {train.shape}   new: {new.shape}   share earning > 50K: {train['income_over_50k'].mean():.2f}")
train.head()

train: (6000, 41)   new: (2000, 40)   share earning > 50K: 0.24


,id,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,...,v17,v18,v19,v20,v21,v22,v23,v24,v25,income_over_50k
0,104207,62,Private,174711,HS-grad,9,Never-married,Adm-clerical,Not-in-family,White,...,39,6.135,D18,B19,H20,E21,E22,D23,C24,0
1,104579,53,Private,283320,Bachelors,13,Divorced,Tech-support,Not-in-family,White,...,70,8.940,B18,A19,F20,D21,F22,F23,D24,0
2,103779,25,Private,789600,HS-grad,9,Never-married,Adm-clerical,Unmarried,White,...,24,10.930,F18,B19,E20,C21,F22,F23,A24,0
3,101072,37,Private,139770,HS-grad,9,Never-married,Adm-clerical,Not-in-family,White,...,20,14.070,C18,D19,F20,I21,H22,E23,E24,0
4,102774,22,Private,202153,Some-college,10,Never-married,Adm-clerical,Own-child,White,...,63,1.237,E18,B19,F20,J21,B22,F23,E24,0


### 2.1 · The data card: what each column is for

| Role | Columns | Notes |
|---|---|---|
| **Target** | `income_over_50k` | 0/1, about **24%** are 1 (1,436 of 6,000). Imbalanced, but less than you might think. |
| **Identifier** | `id` | unique in both files; never a feature. Your predictions are saved by `id`. |
| **Numeric features** | `age`, `education_num`, `capital_gain`, `capital_loss`, `hours_per_week`, `v01`–`v18` | |
| **Categorical features** | `workclass`, `marital_status`, `occupation`, `relationship`, `native_country`, `v19`–`v25` | |
| **Sensitive attributes** | `race`, `sex` | your pipeline audits **one** (`data.sensitive_attr`). Choose one (e.g. `sex`) and decide whether the other may be a feature. Write down your choice and why. |
| **Not a feature** | `fnlwgt` | a census *sampling weight* (how many people this row represents), not a characteristic of the person. |
| **Redundant** | `education` | the same information as `education_num`, one to one (`"Bachelors"` is always 13). Keep one: `education_num` keeps the order. |
| **Undocumented** | `v01`–`v25` | 25 anonymised extra columns; nobody knows what they mean. **Don't decide by hand:** keep them as features and let your feature selection decide. What it keeps is one of today's questions. |

<div style="background-color:#fff8e1; border-left:6px solid #f9a825; padding:10px 14px; border-radius:4px">⚠️ A column that isn't listed in <code>numeric_features</code> or <code>categorical_features</code> is <b>silently left out</b> of the model, not dropped with an error. Check that every column you want really reaches the model (the "features kept" line at the end of a run is a good check).</div>

### 2.2 · The diagnosis: what needs cleaning

Already checked for you. Turn each line into a **config setting** (`diagnostics`, `preprocessing`), not into code that names Adult's columns.

| Issue | Where | What it means | What to do |
|---|---|---|---|
| **Placeholder `?`** | `workclass` (379 rows), `occupation` (381), `native_country` (86) | `?` means "unknown": it's a missing value written as text | turn it into a real missing value in cleaning |
| **The same people are missing twice** | the 379 `?` in `workclass` are exactly the `?` in `occupation` (+ 2 people who never worked) | these people earn > 50K about **half as often** (12% vs 25%): being missing *carries information* | impute **and/or** add a missingness flag |
| **Missing, no pattern** | `native_country` `?` (23% vs 24% earn > 50K); empty cells in `v01`–`v06` (about 3% each) | no relation with the target or other columns | impute only. Empty cells are already read as missing by `read_csv` |
| **Spikes that are real** | `capital_gain`: 92% are 0, and 99999 appears 29 times (all earn > 50K) | 99999 is the census's top code (gains "99999 or more"), not an error | **don't** make it missing. Heavy skew: an option to think about, not a rule to apply |
| **Validity rules** | `age` 17–90, `hours_per_week` 1–99, `capital_gain`/`capital_loss` ≥ 0 | no violations in these files, but rules protect future data. A 17-year-old is valid here (74 people) | write Adult's own rules: COMPAS's `age: {min: 18}` would erase all 74 |
| **Many rare categories** | `native_country`: 41 values, 90% `United-States`, many with fewer than 5 people | | think about which encoder suits this (one-hot → about 40 columns) |
| **Duplicates** | none (no repeated rows, no repeated `id`) | | nothing to do; `drop_duplicate_rows()` can stay |
| **New file** | `adult_new.csv` has the same gaps and placeholders | | cleaning must run on it unchanged, and **every row needs a prediction**: never drop rows from it |

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:10px 14px; border-radius:4px">✅ <b>Check after cleaning:</b> run <code>clean_dataset()</code> with your new config and look at <code>value_counts(dropna=False)</code> for <code>workclass</code>, <code>occupation</code> and <code>native_country</code>. If a category called <code>"?"</code> is still there, your config isn't doing what you think: look at <i>which</i> columns <code>_canonicalize_categories()</code> actually touches. Expected missing counts: 379, 381 and 86.</div>

### 2.3 · Main steps

1. **New config, `config_adult.yaml`.** Copy `config.yaml` and change what is COMPAS-specific: `data` (path, target, sensitive attribute, columns to drop), `diagnostics` (from §2.2), `preprocessing` (feature lists, missingness flags), `output`. The search spaces and CV settings can stay.
2. **Let `main.py` take the config as an argument**, defaulting to `config.yaml`.
3. **Run it and fix what breaks.** Each error is a place that only worked for COMPAS. Fix it in a general way, so **both** configs run. Where to look:
   - anything in `src/` that names a COMPAS column;
   - the fairness report;
   - cleaning steps that only act on some columns.
4. **Reach the target scores (§2.4).** Use the scoreboard and the corrected test to compare your runs on the same folds. Which features does selection keep? Any `v` columns?
5. **Predict the new people.** After the final refit, predict `adult_new.csv`: same cleaning (but no `drop_duplicate_rows`), same feature split (there is no target), the refit pipeline's probabilities, a 0/1 prediction. Save `id`, probability and prediction to a CSV in your repo. **Rows out = rows in (2,000).**

<div style="background-color:#fff8e1; border-left:6px solid #f9a825; padding:10px 14px; border-radius:4px">⏱️ Tuning <i>with</i> feature selection on this data takes several minutes per run. Get everything working with tuning off first.</div>

### 2.4 · 🎯 Target scores

Your pipeline should reach **all three in the same run** (cross-validated on the development set, as your pipeline reports them; the nested scores if tuning is on):

| Metric | Target | No model (always "≤ 50K") | Default decision tree |
|---|---|---|---|
| ROC-AUC | **≥ 0.90** | 0.50 | 0.72 |
| Accuracy | **≥ 0.855** | 0.76 | 0.80 |
| F1 (class 1, threshold 0.5) | **≥ 0.66** | 0.00 | 0.58 |

- The targets need a well-built pipeline (good cleaning, sensible features, a good model), and **you can do better** on every one of them.
- Accuracy looks high, but predicting "≤ 50K" for everyone already gets 0.76: always read it next to the floor.
- The metrics pull in different directions: weighting the classes, for example, raises F1 but lowers accuracy. That's why they're asked together.
- The scores alone don't prove your cleaning is right: run the check in §2.2 too.

## 3 · Option B · New kind of target: three classes

**The question:** predict COMPAS's own risk category, `score_text`: **Low** (about 54% of people), **Medium** (27%) or **High** (20%), from the same features. Same data, same cleaning; only the target changes. (The spelling variants, `low` / `LOW` / ..., are already handled by your `canonical_categories`.)

### 3.1 · What a yes/no pipeline assumes, and what changes with three classes

| Part of the pipeline | Binary (now) | Three classes |
|---|---|---|
| Probabilities | `predict_proba(X)[:, 1]`: one number per row | the whole matrix: 3 numbers per row, summing to 1 |
| Prediction | class 1 if P(1) ≥ 0.5 (a threshold you can move) | the class with the highest probability (`argmax`); no single threshold |
| Metrics | `precision`, `recall`, `f1`, `roc_auc`, `average_precision` | averaged over classes: `f1_macro`, `precision_macro`, `recall_macro`, `roc_auc_ovr` (each class vs the rest) |
| Confusion matrix | 2 × 2 (TN, FP, FN, TP) | 3 × 3: no single "false positive" |
| Fairness (false positive rate) | people who didn't reoffend but were flagged | you must define it, e.g. "people not truly High but predicted High", per group |
| Target encoding | one column per category | one column per category **per class** (can just drop this) |
| Labels | 0 / 1 | text labels; XGBoost needs 0, 1, 2 |
| Unchanged | stratified folds, `class_weight`, the dummy floor, tuning, nested CV, the corrected test, the scoreboard | (same) |

### 3.2 · Main steps

1. **New config, `config_multiclass.yaml`**, with `score_text` as the target. Think carefully about what must **not** be a feature:
   - `decile_score` is the same score as a number (1–10): using it gives the answer away;
   - 🤔 and `two_year_recid`? Was it known *when the score was given*?
2. **Let `main.py` take the config as an argument**, defaulting to `config.yaml`.
3. **Go through the table in §3.1 and fix each binary assumption**, in a way that still works for two classes (the COMPAS run must not change). Places to check:
   - the metrics in the config (`cv.metrics`, `selection_metric`) and how scorers are built;
   - the probabilities in `cross_validate_pipeline()` and `nested_cross_validate()`;
   - the out-of-fold report (threshold, confusion matrix) and the fairness report;
   - the target encoder;
   - `VoteSelector`: its L1 voter reads `lasso.coef_[0]`. With three classes there is one row of coefficients **per class**: what should "the feature was used" mean then?
   - the labels, for XGBoost.
4. **Compare models and tuning as usual**, on `roc_auc_ovr` and macro F1. Which class is hardest to predict, and why?

### 3.3 · 🎯 Target scores

All three in the same run, cross-validated on the development set (add `accuracy` and `f1_macro` to `cv.metrics`; `roc_auc_ovr` is the selection metric):

| Metric | Target | No model (always "Low") | Default decision tree |
|---|---|---|---|
| ROC-AUC (one vs rest) | **≥ 0.75** | 0.50 | 0.63 |
| Accuracy | **≥ 0.63** | 0.54 | 0.58 |
| Macro F1 | **≥ 0.53** | 0.23 | 0.49 |

- You can do a little better than these, but not much: these features plateau, just like in the binary task. **The real work in option B is the code, not the score.**
- Macro F1 gives the three classes equal weight, so a model that ignores Medium can have decent accuracy and a poor macro F1.

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:10px 14px; border-radius:4px">✅ <b>Check:</b> one quick way to know a function now handles both cases is to run it on both configs. A good fix makes the binary COMPAS run give <b>exactly</b> the same numbers as before today.</div>

## 4 · README task (due Saturday 23:59)

- Add the W9 row to "Pipeline progress".
- A new section, **"Beyond COMPAS"** (option A or B):
  - how to run your new config;
  - **what broke, why, and how you fixed it** (a short table is fine);
  - your results table: the runs you compared, and the one that reaches the target scores (option A: also the features kept);
  - 3–5 sentences: what was the hardest assumption to find? What got you to the target scores? For B: which class is hardest, and why?
- Option A: the predictions file for `adult_new.csv` in your repo.

**For the weekly check:** your option running with its own config and reaching the target scores and the README section.